In [1]:
import sys

print("Python executable used by this notebook:")
print(sys.executable)
print(sys.version)


!{sys.executable} -m pip install --user ultralytics opencv-python pandas

Python executable used by this notebook:
/usr/local/anaconda3/bin/python
3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]


In [2]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import os
import csv

model = YOLO("yolov8n.pt")

def detect_person(image_path, model, conf_threshold=0.5, min_box_area=0, save_output=False, output_dir="output_images"):
    image_path = str(image_path)

    if not os.path.exists(image_path):
        raise FileNotFoundError(f"{image_path} does not exist")

    results = model.predict(source=image_path, conf=conf_threshold, verbose=False)

    human_present = False
    best_person_conf = 0.0
    person_count = 0

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            class_name = model.names[cls_id]

            x1, y1, x2, y2 = box.xyxy[0].tolist()
            area = max(0, x2 - x1) * max(0, y2 - y1)

            if class_name == "person" and conf >= conf_threshold and area >= min_box_area:
                human_present = True
                person_count += 1
                if conf > best_person_conf:
                    best_person_conf = conf

        if save_output:
            os.makedirs(output_dir, exist_ok=True)
            drawn = r.plot()
            stem = Path(image_path).stem
            out_path = os.path.join(output_dir, f"detected_{stem}.png")
            cv2.imwrite(out_path, drawn)

    return human_present, best_person_conf, person_count


def run_on_folder(folder_path, model, conf_threshold=0.5, min_box_area=0, save_outputs=True, output_dir="output_images"):
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder '{folder_path}' does not exist")

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_files = sorted([f for f in folder.iterdir() if f.suffix.lower() in valid_exts])

    if len(image_files) == 0:
        raise ValueError(f"No image files found inside '{folder_path}'")

    rows = []

    for file in image_files:
        human_present, best_conf, person_count = detect_person(
            image_path=file,
            model=model,
            conf_threshold=conf_threshold,
            min_box_area=min_box_area,
            save_output=save_outputs,
            output_dir=output_dir
        )

        rows.append({
            "filename": file.name,
            "human_present": human_present,
            "best_person_conf": round(best_conf, 4),
            "person_count": person_count
        })

    return rows


def save_results_to_csv(rows, csv_name):
    if not rows:
        print("No rows to save.")
        return

    with open(csv_name, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["filename", "human_present", "best_person_conf", "person_count"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved results to: {csv_name}")

In [3]:
folder_path = "PNGImages"
conf_threshold = 0.50
min_box_area = 0
save_outputs = True
output_dir = "DetectedOutputs"

rows = run_on_folder(
    folder_path=folder_path,
    model=model,
    conf_threshold=conf_threshold,
    min_box_area=min_box_area,
    save_outputs=save_outputs,
    output_dir=output_dir
)

print("Detection results:")
for row in rows:
    print(row)

save_results_to_csv(rows, "person_detection_results.csv")

num_images = len(rows)
num_positive = sum(1 for row in rows if row["human_present"] is True)
num_negative = sum(1 for row in rows if row["human_present"] is False)

print("\nSummary:")
print("Total images:", num_images)
print("Images with person detected:", num_positive)
print("Images with no person detected:", num_negative)
print("Annotated output images saved in:", output_dir)

[W323 16:22:12.147602096 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:12.148435768 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.911818248 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.912472697 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.243796772 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.244435478 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.362502833 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.363143929 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.457021160 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:13.457610993 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:2

Detection results:
{'filename': 'FudanPed00001.png', 'human_present': True, 'best_person_conf': 0.9201, 'person_count': 2}
{'filename': 'FudanPed00002.png', 'human_present': True, 'best_person_conf': 0.8479, 'person_count': 2}
{'filename': 'FudanPed00003.png', 'human_present': True, 'best_person_conf': 0.8698, 'person_count': 1}
{'filename': 'FudanPed00004.png', 'human_present': True, 'best_person_conf': 0.8697, 'person_count': 3}
{'filename': 'FudanPed00005.png', 'human_present': True, 'best_person_conf': 0.8716, 'person_count': 2}
{'filename': 'FudanPed00006.png', 'human_present': True, 'best_person_conf': 0.8581, 'person_count': 3}
{'filename': 'FudanPed00007.png', 'human_present': True, 'best_person_conf': 0.9024, 'person_count': 6}
{'filename': 'FudanPed00008.png', 'human_present': True, 'best_person_conf': 0.8567, 'person_count': 3}
{'filename': 'FudanPed00009.png', 'human_present': True, 'best_person_conf': 0.9062, 'person_count': 3}
{'filename': 'FudanPed00010.png', 'human_pres

[W323 16:22:33.828762198 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.829349395 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.


In [4]:
negative_rows = run_on_folder(
    folder_path="NegativeImages",
    model=model,
    conf_threshold=0.5,
    min_box_area=0,
    save_outputs=True,
    output_dir="NegativeDetectedOutputs"
)

print("Negative image results:")
for row in negative_rows:
    print(row)

true_negatives = sum(1 for row in negative_rows if row["human_present"] is False)
false_positives = sum(1 for row in negative_rows if row["human_present"] is True)
total_negatives = len(negative_rows)

print("\nNegative set summary:")
print("Total negative images:", total_negatives)
print("True Negatives:", true_negatives)
print("False Positives:", false_positives)

if total_negatives > 0:
    print("True Negative Rate:", round(true_negatives / total_negatives, 4))
    print("False Positive Rate:", round(false_positives / total_negatives, 4))

save_results_to_csv(negative_rows, "negative_results.csv")

[W323 16:22:33.925737099 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.926341514 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.006924644 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.007513036 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.088910769 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.089505244 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.256132000 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.256685998 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.264154457 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:33.264704555 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:2

Negative image results:
{'filename': 'img 10.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 11.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 12.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 123.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 1234.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 4.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 43.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 5.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 55.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'img 9.webp', 'human_present': False, 'best_person_conf': 0.0, 'person_count': 0}
{'filename': 'im

In [5]:

positive_folder = "PNGImages"

df_pos = run_on_folder(
    folder_path=positive_folder,
    model=model,
    conf_threshold=0.5,
    min_box_area=0,
    save_outputs=True,
    output_dir="PositiveDetectedOutputs"
)

print("Positive image results:")
for row in df_pos:
    print(row)

true_positives = sum(1 for row in df_pos if row["human_present"] is True)
false_negatives = sum(1 for row in df_pos if row["human_present"] is False)
total_positives = len(df_pos)

print("\nPositive set summary:")
print("Total positive images:", total_positives)
print("True Positives:", true_positives)
print("False Negatives:", false_negatives)

if total_positives > 0:
    print("True Positive Rate:", round(true_positives / total_positives, 4))
    print("False Negative Rate:", round(false_negatives / total_positives, 4))

save_results_to_csv(df_pos, "positive_results.csv")

[W323 16:22:35.905955693 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:35.906556329 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:35.588999112 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:35.589597031 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:36.999864516 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:36.000440402 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:36.114970278 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:36.115573186 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:36.208456148 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:36.209033424 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:2

Positive image results:
{'filename': 'FudanPed00001.png', 'human_present': True, 'best_person_conf': 0.9201, 'person_count': 2}
{'filename': 'FudanPed00002.png', 'human_present': True, 'best_person_conf': 0.8479, 'person_count': 2}
{'filename': 'FudanPed00003.png', 'human_present': True, 'best_person_conf': 0.8698, 'person_count': 1}
{'filename': 'FudanPed00004.png', 'human_present': True, 'best_person_conf': 0.8697, 'person_count': 3}
{'filename': 'FudanPed00005.png', 'human_present': True, 'best_person_conf': 0.8716, 'person_count': 2}
{'filename': 'FudanPed00006.png', 'human_present': True, 'best_person_conf': 0.8581, 'person_count': 3}
{'filename': 'FudanPed00007.png', 'human_present': True, 'best_person_conf': 0.9024, 'person_count': 6}
{'filename': 'FudanPed00008.png', 'human_present': True, 'best_person_conf': 0.8567, 'person_count': 3}
{'filename': 'FudanPed00009.png', 'human_present': True, 'best_person_conf': 0.9062, 'person_count': 3}
{'filename': 'FudanPed00010.png', 'human

[W323 16:22:54.648189680 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W323 16:22:54.648744358 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.


In [6]:
# =========================================
# Final combined evaluation
# =========================================

TP = true_positives
FN = false_negatives
TN = true_negatives
FP = false_positives

total = TP + TN + FP + FN
accuracy = (TP + TN) / total if total > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

print("Final Metrics")
print("TP =", TP)
print("TN =", TN)
print("FP =", FP)
print("FN =", FN)
print("Accuracy =", round(accuracy, 4))
print("Precision =", round(precision, 4))
print("Recall =", round(recall, 4))

Final Metrics
TP = 183
TN = 15
FP = 0
FN = 0
Accuracy = 1.0
Precision = 1.0
Recall = 1.0
